In [ ]:
!pip install pyPDF2

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torchinfo import summary

import torchvision
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np
import random
import time
import re
from torch.utils.data import Dataset, DataLoader

In [ ]:
#changing the pdf file into a .txt file
from PyPDF2 import PdfReader

# nome del file PDF da convertire
pdf_file = "1984.pdf"

# nome del file di output
txt_file = "testo.txt"

# apri il PDF
with open(pdf_file, "rb") as f:
    reader = PdfReader(f)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"

# salva il testo in un file .txt
with open(txt_file, "w", encoding="utf-8") as f:
    f.write(text)

print(f"Conversione completata! File salvato come {txt_file}")

In [ ]:
#reading raw data to create dataset
with open("testo.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read().lower()

# cleaning: keeping only alphanumeric words (letters only, no numbers/punctuation)
cleaned_text = re.sub(r'[^a-zA-Zàèéìòù]+', ' ', raw_text)

#splitting into separate words
final_text = cleaned_text.split()

word_set = sorted(set(final_text))

# creating sets 
stoi = {w: i for i, w in enumerate(word_set)}
itos = {i: w for i, w in enumerate(word_set)}

In [ ]:
#creating the targets for the trainset and validation set
seq_length = 4
data = []

#creating batches of data
for i in range(len(final_text) - seq_length):
    seq = [stoi[w] for w in final_text[i:i+seq_length]]  
    target = stoi[final_text[i + seq_length]]
    data.append((seq, target))

idx_slice = int(len(data)*0.8)

X = [x for x, y in data]
Y = [y for x, y in data]

X_train = X[:idx_slice]
Y_train = Y[:idx_slice]

X_val = X[idx_slice:]
Y_val = Y[idx_slice:]

In [ ]:
#transforming arrays into tensors
X_train = torch.tensor(X_train, dtype = torch.long)
Y_train = torch.tensor(Y_train, dtype = torch.long)

X_val = torch.tensor(X_val, dtype = torch.long)
Y_val = torch.tensor(Y_val, dtype = torch.long)



In [ ]:
#creating the loaders

class WordDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = WordDataset(X_train, Y_train)
val_dataset = WordDataset(X_val, Y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
#architechture of the network

class MlpWordPredictor(nn.Module):
    def __init__(self, vocab_size, embed_dim, seq_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.layer1 = nn.Linear(embed_dim * seq_length, 256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        
        self.layer2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        
        self.layer3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        
        self.layer4 = nn.Linear(64, vocab_size)
       

    def forward(self, x):
            x = self.embedding(x)           
            x = x.view(x.size(0), -1)      
            x = self.layer1(x)
            x = self.relu1(x)
            x = self.dropout1(x)
        
            x = self.layer2(x)
            x = self.relu2(x)
            x = self.dropout2(x)
        
            x = self.layer3(x)
            x = self.relu3(x)
            x = self.dropout3(x)
        
            x = self.layer4(x)
            return x
    

In [ ]:
#defining the traning loop
def train(model, epochs, criterion, optimizer, train_loader, DEVICE):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        running_corrects = 0
        total = 0

        for value, target in train_loader:
            value = value.to(DEVICE)
            target = target.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(value)
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * value.size(0)
            
            # calcolo dell'accuracy
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == target).item()
            total += target.size(0)

        avg_loss = running_loss / total
        accuracy = running_corrects / total * 100
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {accuracy:.2f}%")


#defining the validation loop
def validation(model, criterion, validation_loader, DEVICE):
    model.eval()
    val_loss = 0
    corrects = 0
    total = 0

    with torch.no_grad():
        for value, target in validation_loader:
            value = value.to(DEVICE)
            target = target.to(DEVICE)
            outputs = model(value)
            loss = criterion(outputs, target)
            val_loss += loss.item() * value.size(0)

            # calcolo dell'accuracy
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == target).item()
            total += target.size(0)

    avg_loss = val_loss / total
    accuracy = corrects / total * 100
    print(f"Validation Loss: {avg_loss:.4f} - Accuracy: {accuracy:.2f}%")

In [ ]:
#defining the main function
if __name__ == "__main__":
    
    #defyining the device to run the code on
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", DEVICE)

    #defyining hyperparameters
    learning_rate = 0.001
    epochs = 100
    embed_dim = 50
    batch_size = 32
    vocab_size = len(word_set)
    criterion = nn.CrossEntropyLoss()
    

    #creating the model
    model = MlpWordPredictor(vocab_size, embed_dim, seq_length)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.to(DEVICE)


    #training loop
    train(model, epochs, criterion, optimizer, train_loader, DEVICE)
    validation(model, criterion, val_loader, DEVICE)

In [ ]:
!pip install git